# Profile: how a format plugs in

A profile is the package `lode/profiles/<format>/` that tells LODE how to read and show one format. This notebook inspects the profiles that ship with LODE. To write one, see [Extending LODE](../extending.md).

:::{warning} Beta version · unstable
LODE 2 is under development and in beta: anything described in this documentation may change. This page describes internals that can change without notice.
:::

## Methods to inspect a profile

| Method | What it gives |
|---|---|
| `profiles.names()` | The names of the registered formats; `names(enabled_only=True)` only the enabled ones. |
| `profiles.get(name)` | The `Profile` of a format, or `ValueError` if the format is unknown. |
| `profiles.get_enabled(name)` | The `Profile`, only if the format is enabled: what the CLI and the API ask. |
| `profiles.of(cls)` | The `Profile` of an `Artefact` class, subclasses included. |
| `profiles.artefact_class_named(name)` | The `Artefact` class with that name, such as `"OwlOntology"`. |
| `profile.path()` | The folder of the profile. |
| `profile.artefact_class()` | The `Artefact` class of the profile. |
| `profiles.resolve(path, package)` | The class a `"module:Name"` path points to. |

## Examples

One for each method of the table, on the profile that ships with LODE for OWL.

### `profiles.names()`

In [1]:
from lode import profiles

print(profiles.names())
print(profiles.names(enabled_only=True))

['owl', 'skos']
['owl']


### `profiles.get(name)`

In [2]:
owl = profiles.get("owl")
owl

Profile(name='owl', package='lode.profiles.owl', reader='.reader:OwlReader', viewer='.viewer:OwlViewer', artefact='.artefact:OwlOntology', post_checks='.checks:run_post_checks', enabled=True)

A `Profile` is a small record: the name of the format, its package, and the paths of its reader, viewer and `Artefact` classes. A path that starts with a dot is relative to the package of the profile. `post_checks` is optional, and `enabled` says whether the format can be named by a string, as the CLI and the API do.

### `profiles.get_enabled(name)`

In [3]:
print(profiles.get_enabled("OWL").name)

try:
    profiles.get_enabled("shacl")
except Exception as error:
    print(type(error).__name__, "-", error)

owl
ArtefactValidationError - Format 'shacl' is not available


### `profiles.of(cls)`

In [4]:
from lode import OwlOntology

print(profiles.of(OwlOntology).name)

owl


### `profiles.artefact_class_named(name)`

In [5]:
print(profiles.artefact_class_named("OwlOntology"))

<class 'lode.profiles.owl.artefact.OwlOntology'>


### `profile.path()`

In [6]:
for format in profiles.names():
    folder = profiles.get(format).path()
    print(f"{format:5}", sorted(p.name for p in folder.iterdir() if p.name != "__pycache__"))

owl   ['__init__.py', 'artefact.py', 'checks.py', 'reader.py', 'reader.yaml', 'viewer.py', 'viewer.yaml']
skos  ['__init__.py', 'artefact.py', 'reader.py', 'reader.yaml', 'templates', 'viewer.py', 'viewer.yaml']


### `profile.artefact_class()`

In [7]:
print(owl.artefact_class())

<class 'lode.profiles.owl.artefact.OwlOntology'>


### `profiles.resolve(path, package)`

In [8]:
print(profiles.resolve(owl.reader, owl.package))
print(profiles.resolve("lode.models:Concept"))

<class 'lode.profiles.owl.reader.OwlReader'>
<class 'lode.models.concept.Concept'>


## How LODE uses it

Everything that depends on the format asks the registry: [`lode.read`](../usage/python_library.md) for the reader, `lode.viewer.get_viewer` for the [viewer](../viewer.md), and the [CLI](../usage/cli.md) and the [API](../api/endpoints.md) for the formats they accept.

LODE ships `owl` and `skos`. Other packages add formats through the `lode.profiles` entry point group.

## Writing your own profile

This page only inspects the profiles that ship with LODE. You do not extend one of them: a new format is a profile of its own. To write it, go to [Extending LODE](../extending.md): [Anatomy of a profile](../extending.md#anatomy-of-a-profile) and [Step 3, declare the profile](../extending.md#step-3-declare-the-profile).